In [1]:
import pandas as pd
import numpy as np
path = r"C:\Users\ashle\OneDrive\Desktop\UManitoba\Courses\FALL 2026\COMP 3380\GroupProject\DataCleaning\reviews.csv"

In [2]:
bad_lines = []

def catch_bad_line(line):
    bad_lines.append(line)
    return None

df = pd.read_csv(path, engine='python', on_bad_lines=catch_bad_line)

In [3]:
print(f"Loaded {len(df)} valid rows.")
print(f"Captured {len(bad_lines)} malformed rows for cleaning.")
# Print out the data
df

Loaded 13073 valid rows.
Captured 0 malformed rows for cleaning.


,id,movie_id,author,author_username,author_rating,content,created_at,updated_at,url,movie_title,movie_release_date,movie_rating,content_length
0,58a231c5925141179e000674,11,Cat Ellington,CatEllington,NaN,"(As I'm writing this review, Darth Vader's the...",2017-02-13T22:23:01.268Z,2021-06-23T15:57:53.915Z,https://www.themoviedb.org/review/58a231c59251...,Star Wars,1977-05-25,8.202,3429
1,5d340e7a2f8d090388d21ff2,11,John Chard,John Chard,10.0,A long time ago in a childhood not too far awa...,2019-07-21T07:04:26.939Z,2024-05-16T15:22:23.643Z,https://www.themoviedb.org/review/5d340e7a2f8d...,Star Wars,1977-05-25,8.202,2979
2,5e94ea4566565a0014baa6c5,11,gastyny,gastyny,10.0,Star Wars (1977) is a true masterpiece of cine...,2020-04-13T22:40:05.539Z,2021-06-23T15:58:36.846Z,https://www.themoviedb.org/review/5e94ea456656...,Star Wars,1977-05-25,8.202,564
3,60174e821b70ae003c643834,11,r96sk,r96sk,9.0,A quality start to the franchise.\r\n\r\nI say...,2021-02-01T00:42:42.135Z,2021-06-23T15:58:50.959Z,https://www.themoviedb.org/review/60174e821b70...,Star Wars,1977-05-25,8.202,1269
4,63c2163f05a53300e99bfdd1,11,GenerationofSwine,GenerationofSwine,10.0,Everyone and their mother is going to write re...,2023-01-14T02:41:03.836Z,2023-01-14T02:41:03.886Z,https://www.themoviedb.org/review/63c2163f05a5...,Star Wars,1977-05-25,8.202,2294
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13068,691e10cb8c18b5c8f5de5528,1539104,CinemaSerf,Geronimo1967,6.0,Whilst I do reckon that the cinema is a great ...,2025-11-19T18:47:39.233Z,2025-11-19T18:56:47.184Z,https://www.themoviedb.org/review/691e10cb8c18...,JUJUTSU KAISEN: Execution,2025-11-07,5.628,1192
13069,6906b031e3c847e5e44c85fb,1571470,DogsLoveMe,DogsLoveMe,1.0,When people say only attractive people are bei...,2025-11-02T01:13:21.140Z,2025-11-16T18:39:59.853Z,https://www.themoviedb.org/review/6906b031e3c8...,The Rats: A Witcher Tale,2025-10-29,5.390,193
13070,693cd86a011ca343dcaec713,1571470,jw,jw,2.0,"# LARPing the basic plot\r\n\r\nagain, Netflix...",2025-12-13T03:07:22.722Z,2025-12-13T03:11:20.395Z,https://www.themoviedb.org/review/693cd86a011c...,The Rats: A Witcher Tale,2025-10-29,5.390,1788
13071,6975a3ac2495071dfd10648a,1584215,MovieGuys,MovieGuys,5.0,"""Rather silly"" is my first and lasting reactio...",2026-01-25T05:01:32.390Z,2026-01-25T05:01:32.416Z,https://www.themoviedb.org/review/6975a3ac2495...,The Internship,2026-01-13,6.239,617


In [4]:
# Data overview
df.info()

print("\nDuplicate rows count:\n", df.duplicated().sum())

print("\nNaN/Null values across all columns:\n", df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 13073 entries, 0 to 13072
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  13073 non-null  str    
 1   movie_id            13073 non-null  int64  
 2   author              13073 non-null  str    
 3   author_username     13073 non-null  str    
 4   author_rating       11639 non-null  float64
 5   content             13073 non-null  str    
 6   created_at          13073 non-null  str    
 7   updated_at          13073 non-null  str    
 8   url                 13073 non-null  str    
 9   movie_title         13073 non-null  str    
 10  movie_release_date  13073 non-null  str    
 11  movie_rating        13073 non-null  float64
 12  content_length      13073 non-null  int64  
dtypes: float64(2), int64(2), str(9)
memory usage: 1.3 MB

Duplicate rows count:
 0

NaN/Null values across all columns:
 id                       0
movie_id          

In [5]:
# 1. Fix incorrect data types

## Check the length of all IDs (TMDB hashes are always exactly 24 characters)
print("ID string length breakdown:")
print(df['id'].astype(str).str.len().value_counts())

## Check how many match the valid 24-character hexadecimal format
is_hex = df['id'].astype(str).str.match(r'^[0-9a-fA-F]{24}$')
print(f"\nValid 24-char hex IDs: {is_hex.sum()} / {len(df)}")

print("It seems TMDB IDs are just 24-character, not an incorrect datatype")

ID string length breakdown:
id
24    13073
Name: count, dtype: int64

Valid 24-char hex IDs: 13073 / 13073
It seems TMDB IDs are just 24-character, not an incorrect datatype


In [6]:
# 2. Handle NaN/Null values from author_rating

# Keep author_rating as NaN/Null rather than imputing 0 or mean to avoid skewing calculations (since it's a numeric)
df['author_rating'] = pd.to_numeric(df['author_rating'], errors='coerce') # ensuring it's NaN/Null

In [7]:
# 3. Check date & time
## reference: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
print("Invalid created_at:", pd.to_datetime(df['created_at'], errors='coerce').isna().sum())
print("Invalid updated_at:", pd.to_datetime(df['updated_at'], errors='coerce').isna().sum())
print("There's no invalid datetime formats")

Invalid created_at: 0
Invalid updated_at: 0
There's no invalid datetime formats


In [8]:
## But we want to make this look cleaner by separating the date and the time.

## Convert to datetime objects
created_dt = pd.to_datetime(df['created_at'])
updated_dt = pd.to_datetime(df['updated_at'])

## Extract Date (YYYY-MM-DD) and Time (HH:MM:SS)
df['createdDate'] = created_dt.dt.strftime('%Y-%m-%d')
df['createdTime'] = created_dt.dt.strftime('%H:%M:%S')

df['updatedDate'] = updated_dt.dt.strftime('%Y-%m-%d')
df['updatedTime'] = updated_dt.dt.strftime('%H:%M:%S')

## Drop the old timestamp columnsa
df = df.drop(columns=['created_at', 'updated_at'])

In [9]:
## Preview the newly created columns
print(df[['createdDate', 'createdTime', 'updatedDate', 'updatedTime']].head())

## Checck for missing/null values
print("\nNull counts:")
print(df[['createdDate', 'createdTime', 'updatedDate', 'updatedTime']].isna().sum())

## Verify format patterns (YYYY-MM-DD and HH:MM:SS)
valid_date = df['createdDate'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()
valid_time = df['createdTime'].str.match(r'^\d{2}:\d{2}:\d{2}$').all()
print(f"\nAll createddDate match YYYY-MM-DD: {valid_date}")
print(f"All createdTime match HH:MM:SS: {valid_time}")

## Confirm original columns were removed
print("\nOld columns removed:", 'created_at' not in df.columns and 'updated_at' not in df.columns)

  createdDate createdTime updatedDate updatedTime
0  2017-02-13    22:23:01  2021-06-23    15:57:53
1  2019-07-21    07:04:26  2024-05-16    15:22:23
2  2020-04-13    22:40:05  2021-06-23    15:58:36
3  2021-02-01    00:42:42  2021-06-23    15:58:50
4  2023-01-14    02:41:03  2023-01-14    02:41:03

Null counts:
createdDate    0
createdTime    0
updatedDate    0
updatedTime    0
dtype: int64

All createddDate match YYYY-MM-DD: True
All createdTime match HH:MM:SS: True

Old columns removed: True


In [10]:
# 4. Clean whitespace/escape characters in content

has_whitespace = (df['content'] != df['content'].str.strip()).sum()
has_cr = df['content'].str.contains(r'\r\n').sum()
print(f"Rows with leading/trailing whitespace: {has_whitespace}")
print(f"Rows with \\r\\n newlines: {has_cr}")

Rows with leading/trailing whitespace: 62
Rows with \r\n newlines: 8231


In [11]:
## Strip whitespace and standardize carriage return newlines to standard newlines
df['content'] = df['content'].str.strip().str.replace(r'\r\n', '\n', regex=True)

## Verify result
has_whitespace = (df['content'] != df['content'].str.strip()).sum()
has_cr = df['content'].str.contains(r'\r\n').sum()
print(f"Rows with leading/trailing whitespace: {has_whitespace}")
print(f"Rows with \\r\\n newlines: {has_cr}")

Rows with leading/trailing whitespace: 0
Rows with \r\n newlines: 0


In [12]:
# 5. Extract Reviewer table as a new entity matching ER diagram
## Check user uniqueness (cuz we don't want duplicate reviewer)
unique_reviewers = df['author_username'].nunique()
total_reviews = len(df)
print(f"Total reviews: {total_reviews}, Unique reviewer usernames: {unique_reviewers}")

Total reviews: 13073, Unique reviewer usernames: 1703


In [13]:
## Create Reviewer table (username as PK, author as displayName)
reviewers_df = df[['author_username', 'author']].drop_duplicates(subset=['author_username'])
reviewers_df.rename(columns={'author_username': 'username', 'author': 'displayName'}, inplace=True) # author = displayName ; author_username = username

## Save the Reviewer table to a new CSV file
reviewers_df.to_csv("../reviewer.csv", index=False)

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13073 entries, 0 to 13072
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  13073 non-null  str    
 1   movie_id            13073 non-null  int64  
 2   author              13073 non-null  str    
 3   author_username     13073 non-null  str    
 4   author_rating       11639 non-null  float64
 5   content             13073 non-null  str    
 6   url                 13073 non-null  str    
 7   movie_title         13073 non-null  str    
 8   movie_release_date  13073 non-null  str    
 9   movie_rating        13073 non-null  float64
 10  content_length      13073 non-null  int64  
 11  createdDate         13073 non-null  str    
 12  createdTime         13073 non-null  str    
 13  updatedDate         13073 non-null  str    
 14  updatedTime         13073 non-null  str    
dtypes: float64(2), int64(2), str(11)
memory usage: 1.5 MB


In [17]:
# 6. Final cleanup (renaming current columns)

## Drop the columns not needed for the Review entity
cols_to_drop = [
    'author',               # Kept in Reviewer table as displayName
    'url',                  # Not in ER diagram
    'movie_title',          # Belongs to Movie table
    'movie_release_date',   # Belongs to Movie table
    'movie_rating',         # Belongs to Movie table
    'content_length'        # Not in ER diagram
]

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

## Rename columns to match ER diagram attribute names
df = df.rename(columns={
    'id': 'reviewID',
    'movie_id': 'movieID',
    'author_username': 'username',
    'author_rating': 'rating'
})

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13073 entries, 0 to 13072
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   reviewID     13073 non-null  str    
 1   movieID      13073 non-null  int64  
 2   username     13073 non-null  str    
 3   rating       11639 non-null  float64
 4   content      13073 non-null  str    
 5   createdDate  13073 non-null  str    
 6   createdTime  13073 non-null  str    
 7   updatedDate  13073 non-null  str    
 8   updatedTime  13073 non-null  str    
dtypes: float64(1), int64(1), str(7)
memory usage: 919.3 KB


In [19]:
# Save cleaned data in csv
clean_path = r"C:\Users\ashle\OneDrive\Desktop\UManitoba\Courses\FALL 2026\COMP 3380\GroupProject\DataCleaning\CleanDatas\reviews_cleaned.csv"
df.to_csv(clean_path, index=False)